<a href="https://colab.research.google.com/github/OoiJooYee/ActionDetectionforSignLanguage/blob/main/CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install --upgrade pip
!pip install tensorflow mediapipe scikit-learn opencv-python matplotlib


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 20.8 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.6/35.6 MB 126.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 152.1 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [mediapipe]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydf 0.12.0 requires protobuf<6.0.0,>=5.29.1, but you have protobuf 4.25.8 

In [1]:
import numpy as np
import tensorflow as tf
import cv2
import mediapipe as mp
import sklearn
import matplotlib.pyplot as plt

print("✅ NumPy:", np.__version__)
print("✅ TensorFlow:", tf.__version__)
print("✅ OpenCV:", cv2.__version__)
print("✅ MediaPipe:", mp.__version__)
print("✅ Scikit-learn:", sklearn.__version__)
print("✅ Matplotlib:", plt.matplotlib.__version__)


✅ NumPy: 1.26.4
✅ TensorFlow: 2.18.0
✅ OpenCV: 4.11.0
✅ MediaPipe: 0.10.21
✅ Scikit-learn: 1.6.1
✅ Matplotlib: 3.10.0


In [2]:
print("\n🔧 Testing MediaPipe Holistic Functionality...")
print("=" * 60)
import mediapipe as mp

try:
    # 导入MediaPipe Holistic
    mp_holistic = mp.solutions.holistic
    mp_drawing = mp.solutions.drawing_utils
    mp_drawing_styles = mp.solutions.drawing_styles
    print("✅ MediaPipe Holistic Import Success")

    # 创建Holistic实例
    holistic = mp_holistic.Holistic(
        static_image_mode=False,
        model_complexity=1,
        smooth_landmarks=True,
        enable_segmentation=True,
        smooth_segmentation=True,
        refine_face_landmarks=True,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    )
    print("✅ Holistic creating successfully")

    # 测试处理虚拟图像
    test_image = np.zeros((480, 640, 3), dtype=np.uint8)
    test_image_rgb = cv2.cvtColor(test_image, cv2.COLOR_BGR2RGB)
    results = holistic.process(test_image_rgb)

    holistic.close()
    print("All test pass!")

except Exception as e:
    print(f"❌ Function fail reason: {e}")
    raise


🔧 Testing MediaPipe Holistic Functionality...
✅ MediaPipe Holistic Import Success
✅ Holistic creating successfully
All test pass!


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!pip install -q kaggle
from google.colab import files
files.upload()  # Upload kaggle.json here

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


Saving kaggle.json to kaggle.json


In [5]:
!kaggle datasets download -d risangbaskoro/wlasl-processed
!unzip wlasl-processed.zip -d wlasl_data

Streaming output truncated to the last 5000 lines.
  inflating: wlasl_data/videos/43596.mp4  
  inflating: wlasl_data/videos/43598.mp4  
  inflating: wlasl_data/videos/43599.mp4  
  inflating: wlasl_data/videos/43671.mp4  
  inflating: wlasl_data/videos/43672.mp4  
  inflating: wlasl_data/videos/43674.mp4  
  inflating: wlasl_data/videos/43677.mp4  
  inflating: wlasl_data/videos/43679.mp4  
  inflating: wlasl_data/videos/43680.mp4  
  inflating: wlasl_data/videos/43681.mp4  
  inflating: wlasl_data/videos/43682.mp4  
  inflating: wlasl_data/videos/43684.mp4  
  inflating: wlasl_data/videos/43689.mp4  
  inflating: wlasl_data/videos/43697.mp4  
  inflating: wlasl_data/videos/43698.mp4  
  inflating: wlasl_data/videos/43700.mp4  
  inflating: wlasl_data/videos/43703.mp4  
  inflating: wlasl_data/videos/43726.mp4  
  inflating: wlasl_data/videos/43727.mp4  
  inflating: wlasl_data/videos/43729.mp4  
  inflating: wlasl_data/videos/43730.mp4  
  inflating: wlasl_data/videos/43733.mp4  
  i

In [6]:
#1-------------------------------------
import json
import os
import pandas as pd
import cv2
from tqdm import tqdm

# Load the WLASL metadata file (usually comes with the dataset)
with open('wlasl_data/WLASL_v0.3.json', 'r') as f:
    wlasl_data = json.load(f)

# Create a mapping of video IDs to their gloss labels
video_id_to_label = {}
label_to_index = {}
index = 0

signs_count = 0
for entry in wlasl_data:
    # if signs_count >= 2000:
    #     break

    gloss = entry['gloss']  # The sign word
    if gloss not in label_to_index:
        label_to_index[gloss] = index
        index += 1
        signs_count += 1

    for instance in entry['instances']:
        video_id = instance['video_id']
        video_id_to_label[video_id] = gloss

# Save the mappings for later use
with open('video_id_to_label.json', 'w') as f:
    json.dump(video_id_to_label, f)

with open('label_to_index.json', 'w') as f:
    json.dump(label_to_index, f)

print(f"Created mappings for {len(video_id_to_label)} videos across {len(label_to_index)} unique signs")


Created mappings for 21083 videos across 2000 unique signs


In [7]:
from tqdm import tqdm
import json
import os
import numpy as np
import cv2
import mediapipe as mp

# Load mappings
with open('video_id_to_label.json', 'r') as f:
    video_id_to_label = json.load(f)

with open('label_to_index.json', 'r') as f:
    label_to_index = json.load(f)

mp_holistic = mp.solutions.holistic

def extract_keypoints(results):
    pose = np.array([[res.x, res.y, res.z] for res in results.pose_landmarks.landmark]).flatten() if results.pose_landmarks else np.zeros(33*3)
    lh = np.array([[res.x, res.y, res.z] for res in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(21*3)
    rh = np.array([[res.x, res.y, res.z] for res in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(21*3)
    return np.concatenate([pose, lh, rh])

def process_video(video_path, holistic, max_frames=30):
    cap = cv2.VideoCapture(video_path)
    sequence = []

    while cap.isOpened() and len(sequence) < max_frames:
        ret, frame = cap.read()
        if not ret:
            break

        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False
        results = holistic.process(image)
        image.flags.writeable = True

        keypoints = extract_keypoints(results)
        sequence.append(keypoints)

    cap.release()

    while len(sequence) < max_frames:
        sequence.append(np.zeros(225))  # pad if needed

    return np.array(sequence)

# Process all videos
input_dir = 'wlasl_data/videos'
output_dir = 'keypoints_data'

os.makedirs(output_dir, exist_ok=True)

with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    for video_id, label in tqdm(list(video_id_to_label.items())[:2000]):
        video_path = os.path.join(input_dir, f"{video_id}.mp4")
        if not os.path.exists(video_path):
            continue

        save_dir = os.path.join(output_dir, label)
        os.makedirs(save_dir, exist_ok=True)
        save_path = os.path.join(save_dir, f"{video_id}.npy")

        if os.path.exists(save_path):
            continue

        try:
            sequence = process_video(video_path, holistic)
            np.save(save_path, sequence)
        except Exception as e:
            print(f"Failed to process {video_path}: {e}")

100%|██████████| 2000/2000 [23:04<00:00,  1.44it/s]


In [8]:
#do checking

import os
import numpy as np

keypoints_dir = 'keypoints_data'
broken_files = []

for label in os.listdir(keypoints_dir):
    label_dir = os.path.join(keypoints_dir, label)
    if not os.path.isdir(label_dir):
        continue

    for file in os.listdir(label_dir):
        if file.endswith('.npy'):
            file_path = os.path.join(label_dir, file)
            try:
                data = np.load(file_path)
                if data.shape != (30, 225):
                    print(f"Unexpected shape in {file_path}: {data.shape}")
                    broken_files.append(file_path)
            except Exception as e:
                print(f"Failed to load {file_path}: {e}")
                broken_files.append(file_path)

print(f"\nTotal broken or invalid files: {len(broken_files)}")



Total broken or invalid files: 0


In [10]:
#---Checking-----
import os
import numpy as np

keypoints_dir = 'keypoints_data'
broken_files = []
shape_error_files = []
total_files = 0
valid_files = 0

for label in os.listdir(keypoints_dir):
    label_dir = os.path.join(keypoints_dir, label)
    if not os.path.isdir(label_dir):
        continue

    for file in os.listdir(label_dir):
        if file.endswith('.npy'):
            total_files += 1
            file_path = os.path.join(label_dir, file)
            try:
                data = np.load(file_path)
                if data.shape != (30, 225):
                    print(f"❌ Shape error: {file_path} -> {data.shape}")
                    shape_error_files.append(file_path)
                else:
                    valid_files += 1
            except Exception as e:
                print(f"⚠️ Load failed: {file_path} -> {e}")
                broken_files.append(file_path)

# 打印结果汇总
print("\n====== ✅ 检查完成 ======")
print(f"📦 Total Files: {total_files}")
print(f"✅ Valid Files: {valid_files}")
print(f"❌ Error Shape: {len(shape_error_files)}")
print(f"⚠️ Broken FIles: {len(broken_files)}")


====== ✅ 检查完成 ======
📦 Total Files: 990
✅ Valid Files: 990
❌ Error Shape: 0
⚠️ Broken FIles: 0


In [11]:
#delete the broken files and check what file is broken and error shape
with open('broken_files.txt', 'w') as f:
    for file in broken_files + shape_error_files:
        f.write(file + '\n')

for f in broken_files + shape_error_files:
    os.remove(f)

In [12]:
import os
import json

keypoints_dir = 'keypoints_data'  # where the npy files are saved

labels = sorted(os.listdir(keypoints_dir))
label_to_index = {label: i for i, label in enumerate(labels)}
index_to_label = {str(i): label for label, i in label_to_index.items()}

# Save label mappings
with open('label_to_index.json', 'w') as f:
    json.dump(label_to_index, f)

with open('index_to_label.json', 'w') as f:
    json.dump(index_to_label, f)

# Create video_id to label mappings
video_id_to_label = {}

for label in labels:
    label_path = os.path.join(keypoints_dir, label)
    for file in os.listdir(label_path):
        if file.endswith('.npy'):
            video_id = file.replace('.npy', '')
            video_id_to_label[video_id] = label

with open('video_id_to_label.json', 'w') as f:
    json.dump(video_id_to_label, f)

print(f"Total labels: {len(label_to_index)}")
print(f"Total videos: {len(video_id_to_label)}")


Total labels: 98
Total videos: 990


In [13]:
from collections import defaultdict

label_counts = defaultdict(int)

for label in os.listdir(keypoints_dir):
    label_dir = os.path.join(keypoints_dir, label)
    if os.path.isdir(label_dir):
        count = len([f for f in os.listdir(label_dir) if f.endswith('.npy')])
        label_counts[label] = count

for label, count in label_counts.items():
    print(f"{label}: {count} samples")


paint: 7 samples
last: 12 samples
how: 9 samples
birthday: 6 samples
hot: 10 samples
later: 12 samples
man: 12 samples
give: 10 samples
blue: 8 samples
medicine: 8 samples
what: 12 samples
forget: 7 samples
wrong: 8 samples
color: 8 samples
africa: 9 samples
black: 10 samples
year: 10 samples
time: 8 samples
secretary: 10 samples
cow: 9 samples
dance: 7 samples
doctor: 10 samples
bird: 10 samples
study: 10 samples
thanksgiving: 13 samples
purple: 8 samples
eat: 7 samples
change: 12 samples
all: 8 samples
language: 10 samples
accident: 13 samples
white: 10 samples
need: 6 samples
who: 14 samples
dog: 11 samples
cheat: 10 samples
son: 8 samples
chair: 7 samples
dark: 12 samples
thin: 16 samples
computer: 14 samples
school: 9 samples
cousin: 14 samples
tall: 13 samples
like: 10 samples
paper: 8 samples
fish: 10 samples
full: 10 samples
yes: 12 samples
but: 7 samples
clothes: 5 samples
bowling: 13 samples
candy: 13 samples
decide: 9 samples
hat: 9 samples
fine: 9 samples
jacket: 7 samples


In [14]:
import os
import shutil
from collections import defaultdict

# 原始数据目录
keypoints_dir = 'keypoints_data'

# 输出目录
output_base = 'categorized_keypoints'
os.makedirs(output_base, exist_ok=True)

# 创建子目录
too_few_dir = os.path.join(output_base, 'too_few')
moderate_dir = os.path.join(output_base, 'moderate')
too_many_dir = os.path.join(output_base, 'too_many')

for path in [too_few_dir, moderate_dir, too_many_dir]:
    os.makedirs(path, exist_ok=True)

# 分类并复制
for label in os.listdir(keypoints_dir):
    label_dir = os.path.join(keypoints_dir, label)
    if not os.path.isdir(label_dir):
        continue

    files = [f for f in os.listdir(label_dir) if f.endswith('.npy')]
    count = len(files)

    # 分类
    if count <= 12:
        target_root = too_few_dir
    elif count <= 16:
        target_root = moderate_dir
    else:
        target_root = too_many_dir

    # 创建子文件夹
    target_label_dir = os.path.join(target_root, label)
    os.makedirs(target_label_dir, exist_ok=True)

    # 复制文件
    for f in files:
        src_path = os.path.join(label_dir, f)
        dst_path = os.path.join(target_label_dir, f)
        shutil.copyfile(src_path, dst_path)

    print(f"{label}: {count} samples -> {target_root.split('/')[-1]}")

print("\n✅ Done. Samples have been categorized and copied.")


paint: 7 samples -> too_few
last: 12 samples -> too_few
how: 9 samples -> too_few
birthday: 6 samples -> too_few
hot: 10 samples -> too_few
later: 12 samples -> too_few
man: 12 samples -> too_few
give: 10 samples -> too_few
blue: 8 samples -> too_few
medicine: 8 samples -> too_few
what: 12 samples -> too_few
forget: 7 samples -> too_few
wrong: 8 samples -> too_few
color: 8 samples -> too_few
africa: 9 samples -> too_few
black: 10 samples -> too_few
year: 10 samples -> too_few
time: 8 samples -> too_few
secretary: 10 samples -> too_few
cow: 9 samples -> too_few
dance: 7 samples -> too_few
doctor: 10 samples -> too_few
bird: 10 samples -> too_few
study: 10 samples -> too_few
thanksgiving: 13 samples -> moderate
purple: 8 samples -> too_few
eat: 7 samples -> too_few
change: 12 samples -> too_few
all: 8 samples -> too_few
language: 10 samples -> too_few
accident: 13 samples -> moderate
white: 10 samples -> too_few
need: 6 samples -> too_few
who: 14 samples -> moderate
dog: 11 samples -> to

In [15]:
def count_samples(directory):
    total = 0
    for label in os.listdir(directory):
        label_dir = os.path.join(directory, label)
        if os.path.isdir(label_dir):
            total += len([f for f in os.listdir(label_dir) if f.endswith('.npy')])
    return total

print("\nSummary:")
print(f"Too Few Samples: {count_samples(too_few_dir)}")
print(f"Moderate Samples: {count_samples(moderate_dir)}")
print(f"Too Many Samples: {count_samples(too_many_dir)}")

print("\nLabel Count per Category:")
print(f"Too Few Labels: {len(os.listdir(too_few_dir))}")
print(f"Moderate Labels: {len(os.listdir(moderate_dir))}")
print(f"Too Many Labels: {len(os.listdir(too_many_dir))}")



Summary:
Too Few Samples: 765
Moderate Samples: 225
Too Many Samples: 0

Label Count per Category:
Too Few Labels: 82
Moderate Labels: 16
Too Many Labels: 0


In [60]:
# preprocess_dataset.py
import os
import numpy as np
import json
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

DATA_PATH = 'categorized_keypoints/moderate'  # 使用 moderate 数据就换成 'categorized_keypoints/moderate'
SEQUENCE_LENGTH = 30
FEATURE_DIM = 225

# Step 1: 构建标签映射
labels = sorted(os.listdir(DATA_PATH))
label_map = {label: idx for idx, label in enumerate(labels)}

with open('label_to_index.json', 'w') as f:
    json.dump(label_map, f)

with open('index_to_label.json', 'w') as f:
    json.dump({v: k for k, v in label_map.items()}, f)

# Step 2: 加载序列
sequences, labels_list = [], []
for label in labels:
    label_dir = os.path.join(DATA_PATH, label)
    for file in os.listdir(label_dir):
        if file.endswith('.npy'):
            path = os.path.join(label_dir, file)
            sequence = np.load(path)
            if sequence.shape == (SEQUENCE_LENGTH, FEATURE_DIM):
                sequences.append(sequence)
                labels_list.append(label_map[label])
            else:
                print(f"Skipped invalid file: {file}, shape: {sequence.shape}")

X = np.array(sequences)
y = to_categorical(labels_list)

print(f"X shape: {X.shape}, y shape: {y.shape}")

# Step 3: 划分数据集
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=np.argmax(y, axis=1)
)

print(f"Train shape: {X_train.shape}, {y_train.shape}")
print(f"Validation shape: {X_val.shape}, {y_val.shape}")

# Step 4: 可选 - 保存为 .npy 方便训练用
np.save('X_train.npy', X_train)
np.save('X_val.npy', X_val)
np.save('y_train.npy', y_train)
np.save('y_val.npy', y_val)

print("\n✅ 数据处理完成，保存为 numpy 格式。")


X shape: (225, 30, 225), y shape: (225, 16)
Train shape: (180, 30, 225), (180, 16)
Validation shape: (45, 30, 225), (45, 16)

✅ 数据处理完成，保存为 numpy 格式。


In [ ]:
# sample = np.load(os.path.join(keypoints_dir, label, os.listdir(os.path.join(keypoints_dir, label))[0]))
# print(sample.shape)      # Should be (30, 225)
# print(sample[0][:10])    # Print first frame, first 10 features


(30, 225)
[ 0.50352973  0.261767   -0.68834895  0.52105385  0.22608462 -0.66452163
  0.53086179  0.2256912  -0.66473001  0.53942698]


In [ ]:
# import matplotlib.pyplot as plt

# # Plot the x coordinates of the first 10 landmarks in the first frame
# plt.plot(sample[0][:30:3])  # x values only
# plt.title("X Coordinates of First 10 Keypoints in Frame 0")
# plt.xlabel("Keypoint Index")
# plt.ylabel("X value")
# plt.show()



In [ ]:
# import os
# import json
# import numpy as np

# # 指定固定要保留的类别
# target_classes = ['hello', 'thanks', 'yes', 'no', 'eat', 'drink', 'go', 'come', 'stop', 'help']
# class_to_index = {cls: i for i, cls in enumerate(target_classes)}

# # 加载 wlasl.json
# with open('wlasl.json', 'r') as f:
#     data = json.load(f)

# X = []
# y = []

# for item in data:
#     gloss = item['gloss']
#     if gloss not in target_classes:
#         continue

#     for instance in item['instances']:
#         video_id = instance['video_id']
#         keypoint_path = os.path.join('dataset/keypoints', f'{video_id}.npy')

#         # 确保关键点文件存在
#         if not os.path.exists(keypoint_path):
#             continue

#         keypoints = np.load(keypoint_path)
#         if keypoints.shape != (30, 225):  # 可以视情况修改
#             continue

#         X.append(keypoints)
#         label = np.zeros(len(target_classes))
#         label[class_to_index[gloss]] = 1
#         y.append(label)

# X = np.array(X)
# y = np.array(y)

# print(f"✅ 数据加载完成: X.shape = {X.shape}, y.shape = {y.shape}")


In [ ]:
# import os
# import numpy as np
# from sklearn.model_selection import train_test_split
# from tensorflow.keras.utils import to_categorical

# DATA_PATH = 'keypoints_data'  # change to your actual path
# SEQUENCE_LENGTH = 30         # should match what you extracted
# FEATURE_DIM = 225            # should match keypoint length

# labels = sorted(os.listdir(DATA_PATH))
# label_map = {label: idx for idx, label in enumerate(labels)}

# sequences, labels_list = [], []

# for label in labels:
#     label_dir = os.path.join(DATA_PATH, label)
#     for file in os.listdir(label_dir):
#         if file.endswith('.npy'):
#             path = os.path.join(label_dir, file)
#             sequence = np.load(path)
#             if sequence.shape == (SEQUENCE_LENGTH, FEATURE_DIM):
#                 sequences.append(sequence)
#                 labels_list.append(label_map[label])

# X = np.array(sequences)
# y = to_categorical(labels_list)

# print(f"X shape: {X.shape}, y shape: {y.shape}")


In [62]:
actions = sorted(os.listdir('categorized_keypoints/moderate'))
label_map = {label: idx for idx, label in enumerate(actions)}


In [63]:
from sklearn.model_selection import train_test_split

def test_and_split(X, y, test_size=0.2, random_state=42):
    X_train, X_val, y_train, y_val = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
        stratify=y.argmax(axis=1) if y.shape[1] > 1 else None  # stratify by class index
    )

    """
    Split the dataset into training and validation sets.

    Parameters:
    - X (np.array): Input features of shape (samples, sequence_length, feature_dim)
    - y (np.array): One-hot encoded labels of shape (samples, num_classes)
    - test_size (float): Proportion of the dataset to include in the validation split
    - random_state (int): Seed for reproducibility

    Returns:
    - X_train, X_val, y_train, y_val: Split datasets
    """

    return X_train, X_val, y_train, y_val


In [64]:
X_train, X_val, y_train, y_val = test_and_split(X, y)

print(f"Train shape: {X_train.shape}, {y_train.shape}")
print(f"Validation shape: {X_val.shape}, {y_val.shape}")

Train shape: (180, 30, 225), (180, 16)
Validation shape: (45, 30, 225), (45, 16)


In [65]:
from sklearn.model_selection import train_test_split

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y if y.shape[1] > 1 else None
)

print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}, y_test shape: {y_test.shape}")


X_train shape: (180, 30, 225), X_test shape: (45, 30, 225)
y_train shape: (180, 16), y_test shape: (45, 16)


In [ ]:
# from tensorflow.keras.models import Sequential
# from tensorflow.keras.layers import LSTM, Dense
# # from tensorflow.keras.callbacks import TensorBoard

In [ ]:
# log_dir = os.path.join('Logs')
# tb_callback = TensorBoard(log_dir=log_dir)

In [ ]:
# model.compile(
#     optimizer='adam',
#     loss='categorical_crossentropy',
#     metrics=['accuracy']
# )


In [66]:
#CNN MODEL

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.layers import Dropout

model = Sequential()

# 1D Convolution over time steps (frames)
model.add(Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(30, 225)))
model.add(BatchNormalization())
model.add(MaxPooling1D(pool_size=2))

model.add(Conv1D(filters=128, kernel_size=3, activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling1D(pool_size=2))

model.add(Flatten())

model.add(Dense(128, activation='relu'))
model.add(Dropout(0.3))

model.add(Dense(64, activation='relu'))
model.add(Dropout(0.3))

model.add(Dense(y.shape[1], activation='softmax'))  # Number of classes

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_10 (Conv1D)              │ (None, 28, 64)         │        43,264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_10          │ (None, 28, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_10 (MaxPooling1D) │ (None, 14, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_11 (Conv1D)              │ (None, 12, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_11          │ (None, 12, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_11 (MaxPooling1D) │ (None, 6, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_4 (Flatten)             │ (None, 768)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 128)            │        98,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_16 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_17 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 16)             │         1,040 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 176,464 (689.31 KB)

 Trainable params: 176,080 (687.81 KB)

 Non-trainable params: 384 (1.50 KB)

In [99]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, BatchNormalization, Flatten, Dense, Dropout
from tensorflow.keras.regularizers import l2

# Regularization factor
l2_reg = 0.001

model = Sequential()

# First Conv1D block
model.add(Conv1D(filters=64, kernel_size=3, activation='relu',
                 kernel_regularizer=l2(l2_reg), input_shape=(30, 225)))
model.add(BatchNormalization())
model.add(MaxPooling1D(pool_size=2))
model.add(Dropout(0.2))  # Add dropout after pooling

# Second Conv1D block
model.add(Conv1D(filters=128, kernel_size=3, activation='relu',
                 kernel_regularizer=l2(l2_reg)))
model.add(BatchNormalization())
model.add(MaxPooling1D(pool_size=2))
model.add(Dropout(0.3))  # Dropout after second pooling

# Fully connected layers
model.add(Flatten())
model.add(Dense(128, activation='relu', kernel_regularizer=l2(l2_reg)))
model.add(Dropout(0.4))

model.add(Dense(64, activation='relu', kernel_regularizer=l2(l2_reg)))
model.add(Dropout(0.4))

# Output layer
model.add(Dense(y.shape[1], activation='softmax'))

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()


Model: "sequential_11"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_22 (Conv1D)              │ (None, 28, 64)         │        43,264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_22          │ (None, 28, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_22 (MaxPooling1D) │ (None, 14, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_34 (Dropout)            │ (None, 14, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_23 (Conv1D)              │ (None, 12, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_23          │ (None, 12, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_23 (MaxPooling1D) │ (None, 6, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_35 (Dropout)            │ (None, 6, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_8 (Flatten)             │ (None, 768)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_30 (Dense)                │ (None, 128)            │        98,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_36 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_31 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_37 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_32 (Dense)                │ (None, 16)             │         1,040 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 176,464 (689.31 KB)

 Trainable params: 176,080 (687.81 KB)

 Non-trainable params: 384 (1.50 KB)

In [100]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

model.fit(X_train, y_train, epochs=500,
          validation_data=(X_val, y_val),
          callbacks=[])


Epoch 1/500
6/6 ━━━━━━━━━━━━━━━━━━━━ 10s 847ms/step - accuracy: 0.0548 - loss: 4.9228 - val_accuracy: 0.0667 - val_loss: 3.2513
Epoch 2/500
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.0992 - loss: 4.1498 - val_accuracy: 0.0667 - val_loss: 3.2301
Epoch 3/500
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.1100 - loss: 3.8460 - val_accuracy: 0.1333 - val_loss: 3.2131
Epoch 4/500
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.1205 - loss: 3.5884 - val_accuracy: 0.1778 - val_loss: 3.1995
Epoch 5/500
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.0642 - loss: 3.5229 - val_accuracy: 0.1778 - val_loss: 3.1957
Epoch 6/500
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.1435 - loss: 3.3460 - val_accuracy: 0.1556 - val_loss: 3.1949
Epoch 7/500
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.1272 - loss: 3.3241 - val_accuracy: 0.1111 - val_loss: 3.1944
Epoch 8/500
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.1344 - loss: 3.2366 - val_accuracy: 0.0667 - val_los

In [88]:
#Hybrid

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM, Dense, Dropout, BatchNormalization

model = Sequential([
    # Convolutional layer to learn spatial patterns across keypoints
    Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(30, 225)),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),

    # Optional: Add another Conv1D layer to deepen
    Conv1D(filters=128, kernel_size=3, activation='relu'),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),

    # LSTM to learn temporal sequences
    LSTM(128, return_sequences=True),
    Dropout(0.3),
    LSTM(64),

    # Dense layers for classification
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(y_train.shape[1], activation='softmax')  # num_classes from one-hot
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()


Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_18 (Conv1D)              │ (None, 28, 64)         │        43,264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_18          │ (None, 28, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_18 (MaxPooling1D) │ (None, 14, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_19 (Conv1D)              │ (None, 12, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_19          │ (None, 12, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_19 (MaxPooling1D) │ (None, 6, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ (None, 6, 128)         │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_28 (Dropout)            │ (None, 6, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_25 (Dense)                │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_29 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_26 (Dense)                │ (None, 16)             │         1,040 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 254,928 (995.81 KB)

 Trainable params: 254,544 (994.31 KB)

 Non-trainable params: 384 (1.50 KB)

In [101]:
res = model.predict(X_test)  # 得到预测概率分布

2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 427ms/step


In [102]:
i = 10   # 可以换成其他索引

predicted_label = actions[np.argmax(res[i])]
actual_label = actions[np.argmax(y_test[i])]

print(f"Sample {i} → Predicted: {predicted_label}, Actual: {actual_label}")


Sample 10 → Predicted: accident, Actual: candy


In [103]:
from sklearn.metrics import multilabel_confusion_matrix, accuracy_score

In [104]:
yhat = model.predict(X_test)

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step


In [105]:
ytrue = np.argmax(y_test, axis=1).tolist()
yhat = np.argmax(yhat, axis=1).tolist()

In [106]:
multilabel_confusion_matrix(ytrue, yhat)

array([[[37,  5],
        [ 3,  0]],

       [[38,  5],
        [ 1,  1]],

       [[37,  5],
        [ 2,  1]],

       [[39,  3],
        [ 1,  2]],

       [[41,  2],
        [ 2,  0]],

       [[42,  0],
        [ 3,  0]],

       [[39,  3],
        [ 2,  1]],

       [[40,  2],
        [ 1,  2]],

       [[42,  0],
        [ 3,  0]],

       [[42,  0],
        [ 2,  1]],

       [[42,  0],
        [ 1,  2]],

       [[42,  0],
        [ 3,  0]],

       [[41,  2],
        [ 2,  0]],

       [[42,  0],
        [ 3,  0]],

       [[41,  1],
        [ 3,  0]],

       [[36,  6],
        [ 2,  1]]])

In [107]:
accuracy_score(ytrue, yhat)

0.24444444444444444

In [109]:
# 8. Save the model
import time
model_name = f'/content/drive/MyDrive/cnn_model_{int(time.time())}.keras'
model_name = f'/content/drive/MyDrive/cnn_model_{int(time.time())}.h5'

model.save(model_name)

# Save the class labels mapping
with open('index_to_label.json', 'w') as f:
    json.dump(index_to_label, f)



IGNORE

IGNORE

In [ ]:
# 安装兼容版本的 numpy 和 tensorflow
!pip uninstall -y numpy
!pip install numpy==1.23.5 --force-reinstall
!pip install --upgrade --no-cache-dir tensorflow


In [ ]:
print("\n🧪 测试导入...")
try:
    import numpy as np
    print(f"✅ NumPy 版本: {np.__version__}")

    # 测试 numpy.dtypes 是否可用
    if hasattr(np, 'dtypes'):
        print("✅ NumPy dtypes 可用")
    else:
        print("⚠️  NumPy dtypes 不可用")

except Exception as e:
    print(f"❌ NumPy 错误: {e}")

try:
    import tensorflow as tf
    print(f"✅ TensorFlow 版本: {tf.__version__}")

    # 测试基本功能
    test_tensor = tf.constant([1, 2, 3])
    print(f"✅ TensorFlow 基本功能正常: {test_tensor}")

except Exception as e:
    print(f"❌ TensorFlow 错误: {e}")

try:
    from sklearn.model_selection import train_test_split
    import sklearn
    print(f"✅ scikit-learn 版本: {sklearn.__version__}")
except Exception as e:
    print(f"❌ scikit-learn 错误: {e}")

try:
    from tensorflow.keras.utils import to_categorical
    print("✅ Keras utils 导入成功")

    # 测试 to_categorical 功能
    test_labels = to_categorical([0, 1, 2], num_classes=3)
    print(f"✅ to_categorical 功能正常: {test_labels.shape}")

except Exception as e:
    print(f"❌ Keras utils 错误: {e}")

print("\n" + "="*50)
print("🎯 如果所有测试都通过，你可以继续进行手势识别项目！")
print("🔄 如果还有错误，请重启运行时后再次运行这个代码。")
print("="*50)


In [ ]:
!pip install --upgrade pip
!pip uninstall -y numpy
!pip install numpy==1.23.5
!pip install tensorflow==2.13.0  # 这个版本对 numpy 1.23.5 最稳定

In [ ]:
#2-------------------------------------
def extract_frames(video_path, output_dir, video_id, sample_rate=10):
    """
    Extract frames from a video file
    sample_rate: extract 1 frame per this many frames
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    cap = cv2.VideoCapture(video_path)
    frame_count = 0
    saved_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_count % sample_rate == 0:
            # Preprocess frame
            frame = cv2.resize(frame, (224, 224))  # Resize for CNN
            save_path = os.path.join(output_dir, f"{video_id}_{saved_count:04d}.jpg")
            cv2.imwrite(save_path, frame)
            saved_count += 1

        frame_count += 1

    cap.release()
    return saved_count

# Process the first 2000 videos
videos_dir = 'wlasl_data/videos'
frames_output_dir = 'extracted_frames/'

# Create a dataframe to keep track of processed data
data_info = []

for video_id, label in tqdm(list(video_id_to_label.items())[:500]):
    video_path = os.path.join(videos_dir, f"{video_id}.mp4")

    if os.path.exists(video_path):
        output_subdir = os.path.join(frames_output_dir, label)
        frame_count = extract_frames(video_path, output_subdir, video_id)

        if frame_count > 0:
            data_info.append({
                'video_id': video_id,
                'label': label,
                'frame_count': frame_count,
                'label_index': label_to_index[label]
            })

# Save the dataset info
pd.DataFrame(data_info).to_csv('dataset_info.csv', index=False)

In [ ]:
#3-------------------------------------
import os

frames_dir = 'extracted_frames/'
if os.path.exists(frames_dir):
    # Count directories (labels)
    labels = [d for d in os.listdir(frames_dir) if os.path.isdir(os.path.join(frames_dir, d))]
    print(f"Found {len(labels)} label directories")

    # Count total frames
    total_frames = 0
    for label in labels:
        label_dir = os.path.join(frames_dir, label)
        frames = [f for f in os.listdir(label_dir) if f.endswith(('.jpg', '.png'))]
        total_frames += len(frames)
        print(f"  - {label}: {len(frames)} frames")

    print(f"Total extracted frames: {total_frames}")
else:
    print("Extracted frames directory not found!")

In [ ]:
#4-------------------------------------
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from sklearn.model_selection import train_test_split
import json
import matplotlib.pyplot as plt

# 1. Load your dataset information
df = pd.read_csv('dataset_info.csv')
print(f"Loaded dataset info with {len(df)} entries")

# Get unique labels from the dataset
unique_labels = df['label'].unique()
num_classes = len(unique_labels)
print(f"Found {num_classes} unique classes")

# Create label mappings from scratch
label_to_index = {label: i for i, label in enumerate(unique_labels)}
index_to_label = {str(i): label for i, label in enumerate(unique_labels)}

# Create a direct mapping from video_id to label
video_id_to_label = dict(zip(df['video_id'], df['label']))
video_id_to_label_index = {video_id: label_to_index[label] for video_id, label in video_id_to_label.items()}

# 2. Load the label mapping
with open('label_to_index.json', 'w') as f:
    json.dump(label_to_index, f)

with open('index_to_label.json', 'w') as f:
    json.dump(index_to_label, f)

print("\nVerifying mappings:")
for video_id in list(df['video_id'])[:10]:
    label = video_id_to_label[video_id]
    label_index = label_to_index[label]
    recovered_label = index_to_label[str(label_index)]
    print(f"Video ID: {video_id}, Label: {label}, Index: {label_index}, Recovered Label: {recovered_label}")


In [ ]:
# Split into train and validation
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'] if len(df['label'].unique()) > 1 else None)
train_videos = train_df['video_id'].tolist()
val_videos = val_df['video_id'].tolist()

print(f"\nTraining set: {len(train_videos)} videos")
print(f"Validation set: {len(val_videos)} videos")

# Check if frames exist for a few random videos
print("\nChecking for frames:")
frames_dir = 'extracted_frames'
for video_id in train_videos[:5]:
    # Get label for this video
    label = video_id_to_label[video_id]
    frame_dir = os.path.join(frames_dir, label)

    if os.path.exists(frame_dir):
        frames = [f for f in os.listdir(frame_dir) if f.startswith(str(video_id))]
        print(f"Video {video_id} (label: {label}): {len(frames)} frames")
    else:
        print(f"ERROR: Directory for label {label} not found!")

In [ ]:
# 3. Set up data generators
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 64

# Create a dictionary mapping video IDs to the label index
video_id_to_label_index = dict(zip(df['video_id'], df['label_index']))

# Define a function to generate data from our frames
def frame_generator(video_ids, batch_size):
    while True:
        # Shuffle the video IDs for each epoch
        np.random.shuffle(video_ids)

        for i in range(0, len(video_ids), batch_size):
            batch_ids = video_ids[i:i+batch_size]
            batch_x = []
            batch_y = []

            for video_id in batch_ids:
                # Get label for this video
                label_index = video_id_to_label_index[video_id]

                # Get all frames for this video
                label = index_to_label[str(label_index)]
                frame_dir = os.path.join('extracted_frames', label)
                frames = [f for f in os.listdir(frame_dir) if f.startswith(str(video_id))]

                if frames:
                    # Select a random frame from this video
                    frame_path = os.path.join(frame_dir, np.random.choice(frames))
                    img = tf.keras.preprocessing.image.load_img(
                        frame_path, target_size=IMAGE_SIZE)
                    img_array = tf.keras.preprocessing.image.img_to_array(img)
                    img_array = img_array / 255.0  # Normalize

                    batch_x.append(img_array)
                    batch_y.append(label_index)

            if batch_x:
                yield np.array(batch_x), tf.keras.utils.to_categorical(batch_y, num_classes)


In [ ]:
# Split data into training and validation sets
train_videos, val_videos = train_test_split(df['video_id'].tolist(), test_size=0.2, random_state=42)

# Create generators
train_gen = frame_generator(train_videos, BATCH_SIZE)
val_gen = frame_generator(val_videos, BATCH_SIZE)


In [ ]:
print(df['video_id'].tolist())

In [ ]:
# First, check if we have any data to work with
print(f"Total video IDs available: {len(df)}")
print(f"Training videos: {len(train_videos)}")
print(f"Validation videos: {len(val_videos)}")

# Check if frames directory exists
frames_dir = 'extracted_frames'
if not os.path.exists(frames_dir):
    print(f"ERROR: Frames directory '{frames_dir}' doesn't exist!")
else:
    # Check contents of the frames directory
    labels = [d for d in os.listdir(frames_dir) if os.path.isdir(os.path.join(frames_dir, d))]
    print(f"Found {len(labels)} label directories in {frames_dir}")

    # Check if frames exist for a few random videos
    for video_id in train_videos[:5]:
        # Get label for this video
        label_index = video_id_to_label_index[video_id]
        label = index_to_label[str(label_index)]
        frame_dir = os.path.join(frames_dir, label)

        if os.path.exists(frame_dir):
            frames = [f for f in os.listdir(frame_dir) if f.startswith(str(video_id))]
            print(f"Video {video_id} (label: {label}): {len(frames)} frames")
        else:
            print(f"ERROR: Directory for label {label} not found!")

In [ ]:
# 4. Building the CNN model
from tensorflow.keras.applications import InceptionV3

base_model = InceptionV3(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(1024, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.AdamW(
        learning_rate=0.001,
        weight_decay=0.01,
    ),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# 5. Train the model
steps_per_epoch = len(train_videos) // BATCH_SIZE
validation_steps = max(1, len(val_videos) // BATCH_SIZE)

history = model.fit(
    train_gen,
    steps_per_epoch=steps_per_epoch,
    epochs=30,
    validation_data=val_gen,
    validation_steps=validation_steps
)



In [ ]:
# 6. Plot training history
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
#New CNN Model after batch normalization and decrease the dense
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Using transfer learning with MobileNetV2 for better performance
base_model = InceptionV3(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze the base model
base_model.trainable = False
for layer in base_model.layers[:-30]:
    layer.trainable = True

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(512, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adamW',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# 5. Train the model
steps_per_epoch = len(train_videos) // BATCH_SIZE
validation_steps = max(1, len(val_videos) // BATCH_SIZE)

history = model.fit(
    train_gen,
    steps_per_epoch=steps_per_epoch,
    epochs=30,
    validation_data=val_gen,
    validation_steps=validation_steps,
    callbacks=[early_stop]
)

In [ ]:
print("Number of classes:", num_classes)


In [ ]:
# 6. Plot training history
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# 7. Fine-tune the model by unfreezing some layers
base_model.trainable = True
for layer in base_model.layers[:-10]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_fine = model.fit(
    train_gen,
    steps_per_epoch=steps_per_epoch,
    epochs=30,
    validation_data=val_gen,
    validation_steps=validation_steps
)


In [ ]:
from tensorflow.keras.models import load_model
model = load_model(model_name)


In [ ]:
# 9. Convert to TensorFlow Lite for mobile or edge deployment
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open('sign_language_model.tflite', 'wb') as f:
    f.write(tflite_model)

print("Model training complete and saved!")